# sat_pyc — Satélite PYC (SISE)

Este notebook construye el **satélite PYC**, que consolida la información de personas
del sistema SISE (`axa_col_slv_dv.core_sise`), unificando tres roles
(Tomador, Asegurado, Beneficiario) en una sola tabla.

## Tablas fuente

| Tabla | Qué contiene |
|---|---|
| `ss_mpersona` | Datos personales (nombre, doc, fec_nac, sexo, est_civil, ciiu) |
| `ss_mpersona_telef` | Teléfonos y celulares por persona |
| `ss_mpersona_dir` | Direcciones y emails por persona |
| `ss_mpersona_aut_datos` | Autorización tratamiento de datos (ATDP) |
| `ss_maseg_header` | Header del asegurado (enlace id_persona ↔ cod_aseg) |
| `ss_sg_pv_header` | Header póliza Generales |
| `ss_sv_pv_header` | Header póliza Vida |
| `ss_sg_di_header` | Detalle póliza Generales (asegurados) |
| `ss_sv_di_header` | Detalle póliza Vida (asegurados) |
| `ss_sg_di_benef` | Beneficiarios Generales |
| `ss_sv_pvind_benef` | Beneficiarios Vida |
| `ss_sg_tramo` / `ss_sv_tramo` | Catálogo de ramos |
| `ss_ttipo_doc` | Catálogo tipo de documento |
| `ss_tdpto` | Catálogo departamentos |
| `ss_tmunicipio` | Catálogo municipios |
| `ss_tpais` | Catálogo países |
| `ss_tciiu` | Catálogo actividad económica CIIU |
| `ss_magente` | Catálogo agentes / asesores |

## Tabla destino

`axa_col_slv_dv.stg_cliente.sat_pyc`

## Ejecución — Crear el satélite PYC

La siguiente celda crea (o reemplaza) la tabla `axa_col_slv_dv.stg_cliente.sat_pyc`.
Se aplica deduplicación por `fecha_cargue` (y `nro_endoso` para pólizas) en cada tabla
fuente antes de los JOINs, siguiendo el patrón de las queries de referencia SQL Server.

> ⚠️ **Importante:** Este proceso borra y recrea la tabla completa cada vez que se ejecuta.

In [ ]:
%sql

CREATE OR REPLACE TABLE axa_col_slv_dv.stg_cliente.sat_pyc
USING DELTA AS

WITH

-- ══════════════════════════════════════════════════════════════════════════════
-- BLOQUE 1 — Deduplicación de tablas fuente
-- ══════════════════════════════════════════════════════════════════════════════

mpersona AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id_persona ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_mpersona
    ) t WHERE rn = 1
),

maseg_header AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id_persona, cod_aseg ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_maseg_header
    ) t WHERE rn = 1
),

mpersona_telef AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id_persona, cod_tipo_telef ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_mpersona_telef
    ) t WHERE rn = 1
),

mpersona_dir AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id_persona, cod_tipo_dir ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_mpersona_dir
    ) t WHERE rn = 1
),

mpersona_aut_datos AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id_persona ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_mpersona_aut_datos
    ) t WHERE rn = 1
),

ttipo_doc AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cod_tipo_doc ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_ttipo_doc
    ) t WHERE rn = 1
),

tdpto AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cod_pais, cod_dpto ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_tdpto
    ) t WHERE rn = 1
),

tmunicipio AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cod_pais, cod_dpto, cod_municipio ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_tmunicipio
    ) t WHERE rn = 1
),

tpais AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cod_pais ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_tpais
    ) t WHERE rn = 1
),

tciiu AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cod_ciiu ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_tciiu
    ) t WHERE rn = 1
),

sg_tramo AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cod_ramo ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_sg_tramo
    ) t WHERE rn = 1
),

sv_tramo AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cod_ramo ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_sv_tramo
    ) t WHERE rn = 1
),

-- Pólizas Generales: última fecha_cargue + mayor nro_endoso (excluye cod_grupo_endo = 15)
sg_pv_header AS (
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY cod_suc, cod_ramo, nro_pol
                   ORDER BY fecha_cargue DESC, nro_endoso DESC
               ) AS rn
        FROM axa_col_slv_dv.core_sise.ss_sg_pv_header
        WHERE cod_grupo_endo <> 15
    ) t WHERE rn = 1
),

-- Pólizas Vida: misma lógica
sv_pv_header AS (
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY cod_suc, cod_ramo, nro_pol
                   ORDER BY fecha_cargue DESC, nro_endoso DESC
               ) AS rn
        FROM axa_col_slv_dv.core_sise.ss_sv_pv_header
        WHERE cod_grupo_endo <> 15
    ) t WHERE rn = 1
),

sg_di_header AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id_pv, cod_item, cod_aseg ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_sg_di_header
    ) t WHERE rn = 1
),

sv_di_header AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id_pv, cod_item, cod_aseg ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_sv_di_header
    ) t WHERE rn = 1
),

sg_di_benef AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id_pv, cod_benef ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_sg_di_benef
        WHERE cod_ind_benef = 1
    ) t WHERE rn = 1
),

sv_pvind_benef AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id_pv_cero, cod_aseg ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_sv_pvind_benef
    ) t WHERE rn = 1
),

magente AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cod_agente ORDER BY fecha_cargue DESC) AS rn
        FROM axa_col_slv_dv.core_sise.ss_magente
    ) t WHERE rn = 1
),

-- ══════════════════════════════════════════════════════════════════════════════
-- BLOQUE 2 — Datos de contacto pivotados por persona
-- ══════════════════════════════════════════════════════════════════════════════

-- Email: cod_tipo_dir IN (15, 13)
email AS (
    SELECT
        id_persona,
        MAX(CASE WHEN rnk = 1 THEN txt_direccion END) AS email_1,
        MAX(CASE WHEN rnk = 2 THEN txt_direccion END) AS email_2
    FROM (
        SELECT id_persona, cod_tipo_dir, txt_direccion,
               RANK() OVER (PARTITION BY id_persona, cod_tipo_dir ORDER BY txt_direccion DESC) AS rnk
        FROM mpersona_dir
        WHERE txt_direccion <> '' AND cod_tipo_dir IN (15, 13)
    ) x
    GROUP BY id_persona
),

-- Celular: cod_tipo_telef IN (4, 10)
celular AS (
    SELECT
        id_persona,
        MAX(CASE WHEN rnk = 1 THEN txt_telefono END) AS celular_1,
        MAX(CASE WHEN rnk = 2 THEN txt_telefono END) AS celular_2
    FROM (
        SELECT id_persona, cod_tipo_telef, txt_telefono,
               RANK() OVER (PARTITION BY id_persona, cod_tipo_telef ORDER BY txt_telefono DESC) AS rnk
        FROM mpersona_telef
        WHERE txt_telefono <> '' AND cod_tipo_telef IN (4, 10)
    ) x
    GROUP BY id_persona
),

-- Dirección residencial: cod_tipo_dir NOT IN (15, 13), se toma la de menor código
direccion AS (
    SELECT
        d.id_persona,
        d.txt_direccion,
        d.cod_pais,
        d.cod_dpto,
        d.cod_municipio,
        pa.txt_desc  AS pais,
        dp.txt_desc  AS departamento,
        mu.txt_desc  AS ciudad_residencia
    FROM mpersona_dir d
    LEFT JOIN tpais     pa ON pa.cod_pais = d.cod_pais
    LEFT JOIN tdpto     dp ON dp.cod_pais = d.cod_pais AND dp.cod_dpto = d.cod_dpto
    LEFT JOIN tmunicipio mu ON mu.cod_pais = d.cod_pais AND mu.cod_dpto = d.cod_dpto
                           AND mu.cod_municipio = d.cod_municipio
    WHERE d.cod_tipo_dir NOT IN (15, 13)
      AND d.txt_direccion <> ''
      AND d.cod_tipo_dir = (
              SELECT MIN(x.cod_tipo_dir)
              FROM mpersona_dir x
              WHERE x.id_persona = d.id_persona
                AND x.cod_tipo_dir NOT IN (15, 13)
                AND x.txt_direccion <> ''
          )
),

-- ══════════════════════════════════════════════════════════════════════════════
-- BLOQUE 3 — CTEs por rol (Generales + Vida)
-- ══════════════════════════════════════════════════════════════════════════════

-- ── TOMADOR Generales ────────────────────────────────────────────────────────
-- Tomador: cod_aseg de la póliza coincide con el del header del asegurado

tomador_gen AS (
    SELECT DISTINCT
        p.id_persona,
        m.cod_aseg,
        pv.id_pv,
        pv.cod_suc,
        pv.cod_ramo,
        pv.nro_pol,
        td.txt_desc_redu                                            AS tipo_documento,
        CASE
            WHEN p.cod_tipo_doc = 3 THEN
                CASE
                    WHEN SUBSTRING(COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,''), 1, 1) IN ('8','9')
                    THEN SUBSTRING(COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,''), 1, 9)
                    ELSE COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,'')
                END
            ELSE COALESCE(NULLIF(p.nro_doc,''), p.nro_nit,'')
        END                                                         AS numero_documento,
        p.txt_nombre                                                AS primer_nombre,
        p.txt_apellido1                                             AS primer_apellido,
        NULL                                                        AS segundo_nombre,
        p.txt_apellido2                                             AS segundo_apellido,
        TRIM(CONCAT_WS(' ', p.txt_nombre, p.txt_apellido1, p.txt_apellido2)) AS nombre_completo_razon_social,
        e.email_1                                                   AS correo,
        cel.celular_1                                               AS celular,
        dir.txt_direccion                                           AS direccion_residencial,
        dir.ciudad_residencia,
        dir.departamento,
        dir.pais,
        p.fec_nac                                                   AS fecha_nacimiento,
        p.txt_sexo                                                  AS genero,
        p.cod_est_civil                                             AS estado_civil,
        aut.sn_aut                                                  AS ATDP,
        ci.txt_descr                                                AS actividad_economica,
        CONCAT(pv.cod_suc,'-',pv.cod_ramo,'-',pv.nro_pol)          AS contrato,
        CASE WHEN ra.cod_ttipo_ramo IN (3,10,13,18) THEN 'Colectivo' ELSE 'Individual' END AS tipo_contrato,
        ag.cod_agente                                               AS clave_asesor,
        p.fecha_cargue                                              AS FEC_ACTUALIZACION,
        'TOMADOR'                                                   AS rol
    FROM sg_pv_header pv
    INNER JOIN maseg_header       m   ON m.cod_aseg    = pv.cod_aseg
    INNER JOIN mpersona           p   ON p.id_persona  = m.id_persona
    LEFT  JOIN email              e   ON e.id_persona  = p.id_persona
    LEFT  JOIN celular            cel ON cel.id_persona = p.id_persona
    LEFT  JOIN direccion          dir ON dir.id_persona = p.id_persona
    LEFT  JOIN mpersona_aut_datos aut ON aut.id_persona = p.id_persona
    LEFT  JOIN ttipo_doc          td  ON td.cod_tipo_doc = p.cod_tipo_doc
    LEFT  JOIN sg_tramo           ra  ON ra.cod_ramo    = pv.cod_ramo
    LEFT  JOIN tciiu              ci  ON ci.cod_ciiu    = m.cod_ciiu
    LEFT  JOIN magente            ag  ON ag.cod_agente  = pv.cod_agente
),

-- ── TOMADOR Vida ─────────────────────────────────────────────────────────────

tomador_vid AS (
    SELECT DISTINCT
        p.id_persona, m.cod_aseg, pv.id_pv, pv.cod_suc, pv.cod_ramo, pv.nro_pol,
        td.txt_desc_redu AS tipo_documento,
        CASE
            WHEN p.cod_tipo_doc = 3 THEN
                CASE
                    WHEN SUBSTRING(COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,''), 1, 1) IN ('8','9')
                    THEN SUBSTRING(COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,''), 1, 9)
                    ELSE COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,'')
                END
            ELSE COALESCE(NULLIF(p.nro_doc,''), p.nro_nit,'')
        END AS numero_documento,
        p.txt_nombre AS primer_nombre, p.txt_apellido1 AS primer_apellido,
        NULL AS segundo_nombre, p.txt_apellido2 AS segundo_apellido,
        TRIM(CONCAT_WS(' ', p.txt_nombre, p.txt_apellido1, p.txt_apellido2)) AS nombre_completo_razon_social,
        e.email_1 AS correo, cel.celular_1 AS celular,
        dir.txt_direccion AS direccion_residencial, dir.ciudad_residencia, dir.departamento, dir.pais,
        p.fec_nac AS fecha_nacimiento, p.txt_sexo AS genero, p.cod_est_civil AS estado_civil,
        aut.sn_aut AS ATDP, ci.txt_descr AS actividad_economica,
        CONCAT(pv.cod_suc,'-',pv.cod_ramo,'-',pv.nro_pol) AS contrato,
        CASE WHEN ra.cod_ttipo_ramo IN (3,10,13,18) THEN 'Colectivo' ELSE 'Individual' END AS tipo_contrato,
        ag.cod_agente AS clave_asesor,
        p.fecha_cargue AS FEC_ACTUALIZACION,
        'TOMADOR' AS rol
    FROM sv_pv_header pv
    INNER JOIN maseg_header       m   ON m.cod_aseg    = pv.cod_aseg
    INNER JOIN mpersona           p   ON p.id_persona  = m.id_persona
    LEFT  JOIN email              e   ON e.id_persona  = p.id_persona
    LEFT  JOIN celular            cel ON cel.id_persona = p.id_persona
    LEFT  JOIN direccion          dir ON dir.id_persona = p.id_persona
    LEFT  JOIN mpersona_aut_datos aut ON aut.id_persona = p.id_persona
    LEFT  JOIN ttipo_doc          td  ON td.cod_tipo_doc = p.cod_tipo_doc
    LEFT  JOIN sv_tramo           ra  ON ra.cod_ramo    = pv.cod_ramo
    LEFT  JOIN tciiu              ci  ON ci.cod_ciiu    = m.cod_ciiu
    LEFT  JOIN magente            ag  ON ag.cod_agente  = pv.cod_agente
),

-- ── ASEGURADO (no tomador) Generales ─────────────────────────────────────────
-- Aparece en di_header con cod_aseg distinto al de la póliza

asegurado_gen AS (
    SELECT DISTINCT
        p.id_persona, m.cod_aseg, pv.id_pv, pv.cod_suc, pv.cod_ramo, pv.nro_pol,
        td.txt_desc_redu AS tipo_documento,
        CASE
            WHEN p.cod_tipo_doc = 3 THEN
                CASE
                    WHEN SUBSTRING(COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,''), 1, 1) IN ('8','9')
                    THEN SUBSTRING(COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,''), 1, 9)
                    ELSE COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,'')
                END
            ELSE COALESCE(NULLIF(p.nro_doc,''), p.nro_nit,'')
        END AS numero_documento,
        p.txt_nombre AS primer_nombre, p.txt_apellido1 AS primer_apellido,
        NULL AS segundo_nombre, p.txt_apellido2 AS segundo_apellido,
        TRIM(CONCAT_WS(' ', p.txt_nombre, p.txt_apellido1, p.txt_apellido2)) AS nombre_completo_razon_social,
        e.email_1 AS correo, cel.celular_1 AS celular,
        dir.txt_direccion AS direccion_residencial, dir.ciudad_residencia, dir.departamento, dir.pais,
        p.fec_nac AS fecha_nacimiento, p.txt_sexo AS genero, p.cod_est_civil AS estado_civil,
        aut.sn_aut AS ATDP, ci.txt_descr AS actividad_economica,
        CONCAT(pv.cod_suc,'-',pv.cod_ramo,'-',pv.nro_pol) AS contrato,
        CASE WHEN ra.cod_ttipo_ramo IN (3,10,13,18) THEN 'Colectivo' ELSE 'Individual' END AS tipo_contrato,
        ag.cod_agente AS clave_asesor,
        p.fecha_cargue AS FEC_ACTUALIZACION,
        'ASEGURADO' AS rol
    FROM sg_pv_header pv
    INNER JOIN sg_di_header       di  ON di.id_pv      = pv.id_pv
    INNER JOIN maseg_header       m   ON m.cod_aseg    = di.cod_aseg
    INNER JOIN mpersona           p   ON p.id_persona  = m.id_persona
    LEFT  JOIN email              e   ON e.id_persona  = p.id_persona
    LEFT  JOIN celular            cel ON cel.id_persona = p.id_persona
    LEFT  JOIN direccion          dir ON dir.id_persona = p.id_persona
    LEFT  JOIN mpersona_aut_datos aut ON aut.id_persona = p.id_persona
    LEFT  JOIN ttipo_doc          td  ON td.cod_tipo_doc = p.cod_tipo_doc
    LEFT  JOIN sg_tramo           ra  ON ra.cod_ramo    = pv.cod_ramo
    LEFT  JOIN tciiu              ci  ON ci.cod_ciiu    = m.cod_ciiu
    LEFT  JOIN magente            ag  ON ag.cod_agente  = pv.cod_agente
),

-- ── ASEGURADO (no tomador) Vida ───────────────────────────────────────────────

asegurado_vid AS (
    SELECT DISTINCT
        p.id_persona, m.cod_aseg, pv.id_pv, pv.cod_suc, pv.cod_ramo, pv.nro_pol,
        td.txt_desc_redu AS tipo_documento,
        CASE
            WHEN p.cod_tipo_doc = 3 THEN
                CASE
                    WHEN SUBSTRING(COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,''), 1, 1) IN ('8','9')
                    THEN SUBSTRING(COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,''), 1, 9)
                    ELSE COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,'')
                END
            ELSE COALESCE(NULLIF(p.nro_doc,''), p.nro_nit,'')
        END AS numero_documento,
        p.txt_nombre AS primer_nombre, p.txt_apellido1 AS primer_apellido,
        NULL AS segundo_nombre, p.txt_apellido2 AS segundo_apellido,
        TRIM(CONCAT_WS(' ', p.txt_nombre, p.txt_apellido1, p.txt_apellido2)) AS nombre_completo_razon_social,
        e.email_1 AS correo, cel.celular_1 AS celular,
        dir.txt_direccion AS direccion_residencial, dir.ciudad_residencia, dir.departamento, dir.pais,
        p.fec_nac AS fecha_nacimiento, p.txt_sexo AS genero, p.cod_est_civil AS estado_civil,
        aut.sn_aut AS ATDP, ci.txt_descr AS actividad_economica,
        CONCAT(pv.cod_suc,'-',pv.cod_ramo,'-',pv.nro_pol) AS contrato,
        CASE WHEN ra.cod_ttipo_ramo IN (3,10,13,18) THEN 'Colectivo' ELSE 'Individual' END AS tipo_contrato,
        ag.cod_agente AS clave_asesor,
        p.fecha_cargue AS FEC_ACTUALIZACION,
        'ASEGURADO' AS rol
    FROM sv_pv_header pv
    INNER JOIN sv_di_header       di  ON di.id_pv      = pv.id_pv
    INNER JOIN maseg_header       m   ON m.cod_aseg    = di.cod_aseg
    INNER JOIN mpersona           p   ON p.id_persona  = m.id_persona
    LEFT  JOIN email              e   ON e.id_persona  = p.id_persona
    LEFT  JOIN celular            cel ON cel.id_persona = p.id_persona
    LEFT  JOIN direccion          dir ON dir.id_persona = p.id_persona
    LEFT  JOIN mpersona_aut_datos aut ON aut.id_persona = p.id_persona
    LEFT  JOIN ttipo_doc          td  ON td.cod_tipo_doc = p.cod_tipo_doc
    LEFT  JOIN sv_tramo           ra  ON ra.cod_ramo    = pv.cod_ramo
    LEFT  JOIN tciiu              ci  ON ci.cod_ciiu    = m.cod_ciiu
    LEFT  JOIN magente            ag  ON ag.cod_agente  = pv.cod_agente
),

-- ── BENEFICIARIO Generales ────────────────────────────────────────────────────
-- Beneficiario se une por cod_benef → id_persona en ss_sg_di_benef

beneficiario_gen AS (
    SELECT DISTINCT
        p.id_persona, NULL AS cod_aseg, pv.id_pv, pv.cod_suc, pv.cod_ramo, pv.nro_pol,
        td.txt_desc_redu AS tipo_documento,
        CASE
            WHEN p.cod_tipo_doc = 3 THEN
                CASE
                    WHEN SUBSTRING(COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,''), 1, 1) IN ('8','9')
                    THEN SUBSTRING(COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,''), 1, 9)
                    ELSE COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,'')
                END
            ELSE COALESCE(NULLIF(p.nro_doc,''), p.nro_nit,'')
        END AS numero_documento,
        p.txt_nombre AS primer_nombre, p.txt_apellido1 AS primer_apellido,
        NULL AS segundo_nombre, p.txt_apellido2 AS segundo_apellido,
        TRIM(CONCAT_WS(' ', p.txt_nombre, p.txt_apellido1, p.txt_apellido2)) AS nombre_completo_razon_social,
        e.email_1 AS correo, cel.celular_1 AS celular,
        dir.txt_direccion AS direccion_residencial, dir.ciudad_residencia, dir.departamento, dir.pais,
        p.fec_nac AS fecha_nacimiento, p.txt_sexo AS genero, p.cod_est_civil AS estado_civil,
        aut.sn_aut AS ATDP, ci.txt_descr AS actividad_economica,
        CONCAT(pv.cod_suc,'-',pv.cod_ramo,'-',pv.nro_pol) AS contrato,
        CASE WHEN ra.cod_ttipo_ramo IN (3,10,13,18) THEN 'Colectivo' ELSE 'Individual' END AS tipo_contrato,
        NULL AS clave_asesor,
        p.fecha_cargue AS FEC_ACTUALIZACION,
        'BENEFICIARIO' AS rol
    FROM sg_pv_header pv
    INNER JOIN sg_di_benef        b   ON b.id_pv       = pv.id_pv
    INNER JOIN mpersona           p   ON p.id_persona  = b.cod_benef
    LEFT  JOIN email              e   ON e.id_persona  = p.id_persona
    LEFT  JOIN celular            cel ON cel.id_persona = p.id_persona
    LEFT  JOIN direccion          dir ON dir.id_persona = p.id_persona
    LEFT  JOIN mpersona_aut_datos aut ON aut.id_persona = p.id_persona
    LEFT  JOIN ttipo_doc          td  ON td.cod_tipo_doc = p.cod_tipo_doc
    LEFT  JOIN sg_tramo           ra  ON ra.cod_ramo    = pv.cod_ramo
    LEFT  JOIN tciiu              ci  ON ci.cod_ciiu    = p.cod_ciiu
),

-- ── BENEFICIARIO Vida ─────────────────────────────────────────────────────────
-- Beneficiario Vida se une por id_pv_cero en ss_sv_pvind_benef

beneficiario_vid AS (
    SELECT DISTINCT
        p.id_persona, NULL AS cod_aseg, pv.id_pv, pv.cod_suc, pv.cod_ramo, pv.nro_pol,
        td.txt_desc_redu AS tipo_documento,
        CASE
            WHEN p.cod_tipo_doc = 3 THEN
                CASE
                    WHEN SUBSTRING(COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,''), 1, 1) IN ('8','9')
                    THEN SUBSTRING(COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,''), 1, 9)
                    ELSE COALESCE(NULLIF(p.nro_nit,''), p.nro_doc,'')
                END
            ELSE COALESCE(NULLIF(p.nro_doc,''), p.nro_nit,'')
        END AS numero_documento,
        p.txt_nombre AS primer_nombre, p.txt_apellido1 AS primer_apellido,
        NULL AS segundo_nombre, p.txt_apellido2 AS segundo_apellido,
        TRIM(CONCAT_WS(' ', p.txt_nombre, p.txt_apellido1, p.txt_apellido2)) AS nombre_completo_razon_social,
        e.email_1 AS correo, cel.celular_1 AS celular,
        dir.txt_direccion AS direccion_residencial, dir.ciudad_residencia, dir.departamento, dir.pais,
        p.fec_nac AS fecha_nacimiento, p.txt_sexo AS genero, p.cod_est_civil AS estado_civil,
        aut.sn_aut AS ATDP, ci.txt_descr AS actividad_economica,
        CONCAT(pv.cod_suc,'-',pv.cod_ramo,'-',pv.nro_pol) AS contrato,
        CASE WHEN ra.cod_ttipo_ramo IN (3,10,13,18) THEN 'Colectivo' ELSE 'Individual' END AS tipo_contrato,
        NULL AS clave_asesor,
        p.fecha_cargue AS FEC_ACTUALIZACION,
        'BENEFICIARIO' AS rol
    FROM sv_pv_header pv
    INNER JOIN sv_pvind_benef     b   ON b.id_pv_cero  = pv.id_pv_cero
    LEFT  JOIN maseg_header       m   ON m.cod_aseg     = b.cod_aseg
    LEFT  JOIN mpersona           p   ON p.id_persona   = m.id_persona
    LEFT  JOIN email              e   ON e.id_persona   = p.id_persona
    LEFT  JOIN celular            cel ON cel.id_persona  = p.id_persona
    LEFT  JOIN direccion          dir ON dir.id_persona  = p.id_persona
    LEFT  JOIN mpersona_aut_datos aut ON aut.id_persona  = p.id_persona
    LEFT  JOIN ttipo_doc          td  ON td.cod_tipo_doc = p.cod_tipo_doc
    LEFT  JOIN sv_tramo           ra  ON ra.cod_ramo     = pv.cod_ramo
    LEFT  JOIN tciiu              ci  ON ci.cod_ciiu     = p.cod_ciiu
)

-- ══════════════════════════════════════════════════════════════════════════════
-- BLOQUE 4 — Unificación final de roles
-- ══════════════════════════════════════════════════════════════════════════════

SELECT * FROM tomador_gen
UNION ALL
SELECT * FROM tomador_vid
UNION ALL
SELECT * FROM asegurado_gen
UNION ALL
SELECT * FROM asegurado_vid
UNION ALL
SELECT * FROM beneficiario_gen
UNION ALL
SELECT * FROM beneficiario_vid

## Validación — Verificar el resultado

In [ ]:
%sql
-- Total de filas cargadas por rol
SELECT
    rol,
    COUNT(*) AS total_filas
FROM axa_col_slv_dv.stg_cliente.sat_pyc
GROUP BY rol
ORDER BY rol

In [ ]:
%sql
-- Vista previa de los primeros 5 registros
SELECT
    rol,
    tipo_documento,
    numero_documento,
    primer_nombre,
    primer_apellido,
    nombre_completo_razon_social,
    correo,
    celular,
    ciudad_residencia,
    departamento,
    pais,
    contrato,
    tipo_contrato,
    clave_asesor,
    FEC_ACTUALIZACION
FROM axa_col_slv_dv.stg_cliente.sat_pyc
LIMIT 5